# Hera Kappa Residual Substructure Diagnostics

This notebook compares the saved Hera model convergence map against the FF-SIMS truth convergence map. It is intended to diagnose the non-parametric residual component that remains after the parametric lens model has been fit.

The main residual is defined as

$$
\Delta\kappa(\boldsymbol{\theta}) = \kappa_{\rm truth}(\boldsymbol{\theta}) - \kappa_{\rm model}(\boldsymbol{\theta}).
$$

The notebook computes: a residual map, radial residual variance, residual variance as a function of model convergence, and the isotropic 2D power spectrum of the residual field.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from matplotlib.colors import TwoSlopeNorm
from scipy.ndimage import map_coordinates

try:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name != "lenscluster":
        PROJECT_ROOT = Path("/Users/colinburke/research/lenscluster")
except Exception:
    PROJECT_ROOT = Path("/Users/colinburke/research/lenscluster")

# Set RUN_DIR manually to compare a specific completed run. If left as None,
# the notebook uses the most recently modified Hera stage1 model-kappa FITS file.
RUN_DIR = None

TRUTH_KAPPA_FITS = PROJECT_ROOT / "data/ff_sims/published/hera/kappa_z9_0.fits"
MODEL_KAPPA_FITS_NAME = "truth_recovery_kappa_model_median.fits"
CENTER_ARCSEC = (0.0, 0.0)
N_RADIAL_BINS = 24
N_KAPPA_BINS = 24
N_POWER_BINS = 32

plt.rcParams.update({"figure.dpi": 120})

## Locate Saved Hera Artifacts

The standard truth-recovery pipeline writes `fits/truth_recovery_kappa_model_median.fits` under each stage run directory. The cell below picks the newest Hera stage1 result unless `RUN_DIR` is set explicitly.

In [ ]:
def find_latest_hera_stage1_run(project_root: Path) -> Path:
    pattern = "results/**/hera*/stage1_backprojected_centroid_fit/fits/truth_recovery_kappa_model_median.fits"
    matches = list(project_root.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No Hera model-kappa FITS files found with pattern: {project_root / pattern}")
    latest_model = max(matches, key=lambda path: path.stat().st_mtime)
    return latest_model.parents[1]

run_dir = Path(RUN_DIR) if RUN_DIR is not None else find_latest_hera_stage1_run(PROJECT_ROOT)
model_kappa_fits = run_dir / "fits" / MODEL_KAPPA_FITS_NAME
output_dir = run_dir / "substructure_diagnostics"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT:      {PROJECT_ROOT}")
print(f"RUN_DIR:           {run_dir}")
print(f"truth kappa FITS:  {TRUTH_KAPPA_FITS}")
print(f"model kappa FITS:  {model_kappa_fits}")
print(f"diagnostic output: {output_dir}")

## Load Truth and Model Kappa on the Same Grid

The raw Hera truth map is usually higher resolution than the saved model diagnostic grid. We therefore resample the truth map onto the saved model FITS WCS before forming residuals.

In [ ]:
def load_fits_image_and_wcs(path: Path) -> tuple[np.ndarray, WCS]:
    with fits.open(path) as hdul:
        image = np.asarray(hdul[0].data, dtype=float)
        wcs = WCS(hdul[0].header)
    return image, wcs

def sample_image_on_wcs_grid(image: np.ndarray, image_wcs: WCS, target_wcs: WCS, target_shape: tuple[int, int]) -> np.ndarray:
    y_pixels, x_pixels = np.indices(target_shape, dtype=float)
    ra_deg, dec_deg = target_wcs.pixel_to_world_values(x_pixels, y_pixels)
    source_x, source_y = image_wcs.world_to_pixel_values(ra_deg, dec_deg)
    sampled = map_coordinates(
        image,
        [np.asarray(source_y, dtype=float), np.asarray(source_x, dtype=float)],
        order=1,
        mode="constant",
        cval=np.nan,
    )
    return np.asarray(sampled, dtype=float).reshape(target_shape)

def wcs_arcsec_coordinates(wcs: WCS, shape: tuple[int, int]) -> tuple[np.ndarray, np.ndarray]:
    y_pixels, x_pixels = np.indices(shape, dtype=float)
    ra_deg, dec_deg = wcs.pixel_to_world_values(x_pixels, y_pixels)
    ra0, dec0 = wcs.wcs.crval[:2]
    x_arcsec = (np.asarray(ra_deg, dtype=float) - float(ra0)) * 3600.0 * np.cos(np.deg2rad(float(dec0)))
    y_arcsec = (np.asarray(dec_deg, dtype=float) - float(dec0)) * 3600.0
    return x_arcsec, y_arcsec

truth_native, truth_wcs = load_fits_image_and_wcs(TRUTH_KAPPA_FITS)
model_kappa, model_wcs = load_fits_image_and_wcs(model_kappa_fits)
truth_kappa = sample_image_on_wcs_grid(truth_native, truth_wcs, model_wcs, model_kappa.shape)
x_arcsec, y_arcsec = wcs_arcsec_coordinates(model_wcs, model_kappa.shape)

residual = truth_kappa - model_kappa
fractional_residual = residual / truth_kappa
valid = np.isfinite(truth_kappa) & np.isfinite(model_kappa) & np.isfinite(residual)

print(f"truth native shape: {truth_native.shape}")
print(f"model shape:        {model_kappa.shape}")
print(f"valid pixels:       {valid.sum()} / {valid.size}")
print(f"residual median:    {np.nanmedian(residual):.4g}")
print(f"residual std:       {np.nanstd(residual):.4g}")

## 1. Residual Map

This map shows the spatial structure left over after subtracting the parametric model from the N-body truth map.

In [ ]:
def map_extent(x_arcsec: np.ndarray, y_arcsec: np.ndarray) -> tuple[float, float, float, float]:
    return (
        float(np.nanmin(x_arcsec)),
        float(np.nanmax(x_arcsec)),
        float(np.nanmin(y_arcsec)),
        float(np.nanmax(y_arcsec)),
    )

extent = map_extent(x_arcsec, y_arcsec)
resid_scale = np.nanpercentile(np.abs(residual[valid]), 98.0)

fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.5), constrained_layout=True)

im0 = axes[0].imshow(truth_kappa, origin="lower", extent=extent, cmap="magma", vmin=0.0, vmax=np.nanpercentile(truth_kappa[valid], 99.0))
axes[0].set_title(r"$\kappa_{\rm truth}$")
fig.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(model_kappa, origin="lower", extent=extent, cmap="magma", vmin=0.0, vmax=np.nanpercentile(truth_kappa[valid], 99.0))
axes[1].set_title(r"$\kappa_{\rm model}$")
fig.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(
    residual,
    origin="lower",
    extent=extent,
    cmap="RdBu_r",
    norm=TwoSlopeNorm(vmin=-resid_scale, vcenter=0.0, vmax=resid_scale),
)
axes[2].set_title(r"$\Delta\kappa=\kappa_{\rm truth}-\kappa_{\rm model}$")
fig.colorbar(im2, ax=axes[2], fraction=0.046)

for ax in axes:
    ax.set_xlabel("x [arcsec]")
axes[0].set_ylabel("y [arcsec]")

residual_map_path = output_dir / "hera_kappa_residual_map.png"
fig.savefig(residual_map_path, dpi=180, bbox_inches="tight")
plt.show()
print(residual_map_path)

## 2. Radial Residual Variance

If the residual field is stationary, its variance should not depend strongly on cluster-centric radius. A strong radial trend is evidence for a nonstationary perturbation amplitude.

In [ ]:
def binned_statistic_1d(x: np.ndarray, y: np.ndarray, bins: np.ndarray) -> pd.DataFrame:
    rows = []
    for i, (lo, hi) in enumerate(zip(bins[:-1], bins[1:], strict=True)):
        mask = np.isfinite(x) & np.isfinite(y) & (x >= lo) & (x < hi)
        values = y[mask]
        rows.append(
            {
                "bin_index": i,
                "x_min": lo,
                "x_max": hi,
                "x_center": 0.5 * (lo + hi),
                "count": int(values.size),
                "mean": float(np.nanmean(values)) if values.size else np.nan,
                "median": float(np.nanmedian(values)) if values.size else np.nan,
                "variance": float(np.nanvar(values)) if values.size else np.nan,
                "std": float(np.nanstd(values)) if values.size else np.nan,
                "mad": float(1.4826 * np.nanmedian(np.abs(values - np.nanmedian(values)))) if values.size else np.nan,
            }
        )
    return pd.DataFrame(rows)

x0, y0 = CENTER_ARCSEC
radius = np.sqrt((x_arcsec - x0) ** 2 + (y_arcsec - y0) ** 2)
radial_bins = np.linspace(0.0, np.nanpercentile(radius[valid], 99.5), N_RADIAL_BINS + 1)
radial_df = binned_statistic_1d(radius[valid], residual[valid], radial_bins)
radial_df.to_csv(output_dir / "radial_residual_variance.csv", index=False)

fig, ax = plt.subplots(figsize=(6.2, 4.4))
ax.plot(radial_df["x_center"], radial_df["variance"], marker="o")
ax.set_xlabel("radius [arcsec]")
ax.set_ylabel(r"Var$(\Delta\kappa)$")
ax.set_title("Radial residual variance")
ax.grid(alpha=0.25)
fig.tight_layout()
path = output_dir / "radial_residual_variance.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
plt.show()
radial_df.head(), path

## 3. Residual Variance Versus Model Kappa

This tests whether the amplitude of the missing structure grows in high-convergence regions. If it does, a GP with spatially varying amplitude tied to the smooth model may be more appropriate than a stationary field.

In [ ]:
kappa_values = model_kappa[valid]
residual_values = residual[valid]
kappa_lo, kappa_hi = np.nanpercentile(kappa_values, [1.0, 99.0])
kappa_bins = np.linspace(kappa_lo, kappa_hi, N_KAPPA_BINS + 1)
kappa_df = binned_statistic_1d(kappa_values, residual_values, kappa_bins)
kappa_df.to_csv(output_dir / "residual_variance_vs_model_kappa.csv", index=False)

fig, ax = plt.subplots(figsize=(6.2, 4.4))
ax.plot(kappa_df["x_center"], kappa_df["variance"], marker="o")
ax.set_xlabel(r"$\kappa_{\rm model}$")
ax.set_ylabel(r"Var$(\Delta\kappa)$")
ax.set_title("Residual variance versus model convergence")
ax.grid(alpha=0.25)
fig.tight_layout()
path = output_dir / "residual_variance_vs_model_kappa.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
plt.show()
kappa_df.head(), path

## 4. Isotropic 2D Residual Power Spectrum

This estimates the scale dependence of the residual field. The normalization is most useful for comparing runs analyzed with the same grid and mask; the slope and scale dependence are the primary diagnostics here.

In [ ]:
def pixel_scale_arcsec(wcs: WCS) -> float:
    scales_deg = proj_plane_pixel_scales(wcs) * 3600.0
    return float(np.sqrt(np.abs(scales_deg[0] * scales_deg[1])))

def isotropic_power_spectrum(image: np.ndarray, valid_mask: np.ndarray, wcs: WCS, n_bins: int = 32) -> pd.DataFrame:
    data = np.asarray(image, dtype=float).copy()
    mask = np.asarray(valid_mask, dtype=bool) & np.isfinite(data)
    data[~mask] = np.nan
    data = data - np.nanmean(data)
    data = np.nan_to_num(data, nan=0.0)

    ny, nx = data.shape
    wy = np.hanning(ny)
    wx = np.hanning(nx)
    window = np.outer(wy, wx)
    window_norm = np.mean(window**2)
    data_windowed = data * window

    dx = pixel_scale_arcsec(wcs)
    fft = np.fft.fft2(data_windowed)
    power = (np.abs(fft) ** 2) * (dx**2) / max(window_norm * nx * ny, 1.0)

    kx = np.fft.fftfreq(nx, d=dx)
    ky = np.fft.fftfreq(ny, d=dx)
    kx_grid, ky_grid = np.meshgrid(kx, ky)
    k = np.sqrt(kx_grid**2 + ky_grid**2)

    finite = np.isfinite(k) & np.isfinite(power) & (k > 0)
    k_values = k[finite]
    p_values = power[finite]
    bins = np.geomspace(np.nanmin(k_values), np.nanmax(k_values), n_bins + 1)

    rows = []
    for i, (lo, hi) in enumerate(zip(bins[:-1], bins[1:], strict=True)):
        sel = (k_values >= lo) & (k_values < hi)
        values = p_values[sel]
        rows.append(
            {
                "bin_index": i,
                "k_min_cycle_per_arcsec": lo,
                "k_max_cycle_per_arcsec": hi,
                "k_center_cycle_per_arcsec": np.sqrt(lo * hi),
                "count": int(values.size),
                "power_mean": float(np.nanmean(values)) if values.size else np.nan,
                "power_median": float(np.nanmedian(values)) if values.size else np.nan,
            }
        )
    return pd.DataFrame(rows)

psd_df = isotropic_power_spectrum(residual, valid, model_wcs, n_bins=N_POWER_BINS)
psd_df.to_csv(output_dir / "residual_power_spectrum.csv", index=False)

fig, ax = plt.subplots(figsize=(6.2, 4.4))
ax.loglog(psd_df["k_center_cycle_per_arcsec"], psd_df["power_median"], marker="o", label="median")
ax.loglog(psd_df["k_center_cycle_per_arcsec"], psd_df["power_mean"], marker="s", alpha=0.65, label="mean")
ax.set_xlabel(r"spatial frequency $k$ [cycles arcsec$^{-1}$]")
ax.set_ylabel(r"$P_{\Delta\kappa}(k)$ [arb.] ")
ax.set_title("Isotropic residual power spectrum")
ax.grid(alpha=0.25, which="both")
ax.legend()
fig.tight_layout()
path = output_dir / "residual_power_spectrum.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
plt.show()
psd_df.head(), path

## Suggested Interpretation

- If the residual map is dominated by coherent large-scale patterns, the parametric halo model is missing broad shape/multipole flexibility.
- If the radial residual variance changes strongly with radius, a stationary GP/GRF across the full field is probably too simple.
- If the residual variance grows with `kappa_model`, a nonstationary amplitude such as `A(theta) ∝ kappa_model(theta)^alpha` is worth testing.
- If the power spectrum is close to a power law over useful scales, a spectral GP prior is a natural next model. If it has bumps or scale breaks, the GP kernel/PSD should include those scales explicitly.